# Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Afifa Iqbal
- AI Acknowledgements: Claude was used
- There were a lot of changes made, iteration over the different stop words and all of these have beeen made on the basis of the LDA model which was done repeatedly

## Topic Modeling Analysis

This notebook contains code that performs Topic Modeling Analysis, using LDA, of pre-processed JOB meeting minutes text. 

### Import libraries and lemmatized text data:

In [78]:
# need to ask Anna to remove the years, some common words such as adjourn, order etc. Have to make a list to give to her

In [69]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

# Set up paths, taken from preprocessing notebooks
from pathlib import Path

BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where lemmatized text is stored

path = OUT_DIR / 'lemmatized_text.xz'

# Adapted from UDA Clustering on Text.ipynb
import pickle

with open(path, 'rb') as f:
    documents = pickle.load(f)

print(len(documents))

156


## Cleaning the Lammatized Text 
##### Steps taken:
###### - removing the documents which are empty
###### - removing the digits and numbers (except for years)
###### - creating a custom stop words list and removing those from docs based on the analysis from LDA so far

In [70]:
print([i for i, doc in enumerate(documents) if len(doc) == 0]) #removing the empty docs as well

[34, 43, 44]


In [71]:
# removing the empty doccs
filtered_docs = []
filtered_names = []

for i, doc in enumerate(documents):
    if len(doc) > 0:  # only keep non-empty documents
        filtered_docs.append(doc)
        #filtered_names.append(doc_names[i]) ------  needs to be fixed
        
print (len(filtered_docs))

153


In [72]:
cleaned_docs = []
for doc in filtered_docs:
    cleaned_tokens = [
        word for word in doc
        if not word.isdigit()  # removing numbers as they were showing up as the frequent words in the vocab list
        or (word.isdigit() and len(word) == 4 and word.startswith(('19', '20')))] # keeping years to help with policy questions related to time
    cleaned_docs.append(cleaned_tokens)

In [73]:
# creating a stop words list by reviewing the most frequent words which are not meaningful 
# these words were observed in the initial LDA model 

from collections import Counter
all_words = [word for doc in cleaned_docs for word in doc]
word_counts = Counter(all_words)
#print(word_counts.most_common(300))

In [74]:
#created based on the LDA results + the frequest words observed in the previous step
custom_stop_words = [
    # fillers during conversations - should not be meaningful
    'know', 'think', 'want', 'okay', 'thank',
    'actually', 'thing', 'ask', 'look', 'say', 'yes', 'yeah', 'sure',
    'oh', 'sorry', 'maybe', 'lot', 
    'let', 'bring', 'tell', 'come', 'happen', 'hear',
    'speak', 'try', 'find', 'understand', 'able',
    'way', 'today', 'anybody', 'everybody', 'somebody', 'guy', 'folk',
    'far', 'past', 'long', 'different', 'specific', 'specifically',
    'currently', 'actually', 'forward', 'outside', 'available', 'little',
    'couple', 'ago', 'appreciate', 'mention', 'run', 'add', 'end', 'page',
    
    # some of the single letters are showing up as frequent words
    'e', 't', 's', 'o', 'n', 'r', 'h', 'w', 'c', 'd', 'l', 'g',
    'f', 'p', 'u', 'm', 'y', "'",
    
    # meeting formal words
    'meeting', 'motion', 'second', 'approve', 'adjourn', 'report',
    'member', 'present', 'minute', 'vote', 'board', 'committee',
    'comment', 'public', 'business', 'follow', 'address', 'discussion',
    'record', 'hold', 'begin', 'complete', 'submit', 'review', 'executive', 'president',
    
    # places/locations
    'allegheny', 'county', 'pennsylvania', 'pittsburgh',
    
    # names of specific board numbers maybe? - gets updated
    'hallam', 'evashavik', 'howsie', 'beasom', 'lazzara', 'brinkman',
    'toma', 'bigley', 'innamorato', 'damick', 'wagner', 'moss',
    'klein', 'perkins', 'williams', 'clark', 'bethany', 'ludwig', 'mcdaniel', 'ron', 'dana', 'chuck', 'doug',
    'larry', 'pofi', 'acchs', 'martoni', 'madarino', 'catanese',
    'phillips', 'donna', 'jo', 'joe', 'orlando', 'harper', 'latoya',
    'mullen', 'fitzgerald', 'asturi', 'carol', 'hertz', 'mike',
    'gilmore', 'marion', 'chelsa', 'william', 'dilucente', 'cashman',
    'walker', 'corizon', 'wingard', 'trevor', 'defazio', 'korinski',
    'dilucente', 'right', 'talk', 'man', 'hood', 'price', 'griffin', 'officer', 'connor', 'acj', 'mean', 'zak', 'person', 
                 'visit', 'beth', 'ali', 'kroll', 'davis', 'martin' 
]

In [75]:
#print (ENGLISH_STOP_WORDS) 

#ended up not including these as Anna has done pre-processing so assuming these
#should have been handled there

# if it is decided to add again - still keeping the code here:

#from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
#all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stops


In [76]:
# Source - https://stackoverflow.com/a/46328592
# Posted by PyRsquared
# Retrieved 2026-04-22, License - CC BY-SA 3.0

#removing the custom stop words just created:

stop_words_removed_doc = [
    [word for word in doc if word not in custom_stop_words]
    for doc in cleaned_docs]

In [77]:
# this step is being done (as a debugging step) to understand how many words are being removed due to the custom stop words
# and to figure out if there are unreasonable long docs 

for i, (doc1, doc2) in enumerate(zip(filtered_docs, stop_words_removed_doc)):
    print(f"Document {i+1}: Original = {len(doc1)} words | Cleaned = {len(doc2)} words")
    if len(doc2) > 10000:
        print(f"--------------------------- Document {i+1}: Cleaned = {len(doc2)} words ------------------------------")

Document 1: Original = 350 words | Cleaned = 224 words
Document 2: Original = 493 words | Cleaned = 332 words
Document 3: Original = 485 words | Cleaned = 321 words
Document 4: Original = 545 words | Cleaned = 409 words
Document 5: Original = 261 words | Cleaned = 161 words
Document 6: Original = 694 words | Cleaned = 525 words
Document 7: Original = 499 words | Cleaned = 336 words
Document 8: Original = 763 words | Cleaned = 552 words
Document 9: Original = 506 words | Cleaned = 309 words
Document 10: Original = 679 words | Cleaned = 469 words
Document 11: Original = 357 words | Cleaned = 234 words
Document 12: Original = 377 words | Cleaned = 246 words
Document 13: Original = 642 words | Cleaned = 470 words
Document 14: Original = 506 words | Cleaned = 364 words
Document 15: Original = 846 words | Cleaned = 594 words
Document 16: Original = 590 words | Cleaned = 426 words
Document 17: Original = 787 words | Cleaned = 623 words
Document 18: Original = 727 words | Cleaned = 519 words
D

## Lammatized Text to one String Doc for TF-IDF
###### - Creating a document string for tfidf
###### - Creating another document having only 2000 Tokens to try to fix the issue - tested separately

In [78]:
#joining the tokenized text into a string for tf-idf

docs_string = [' '.join(doc) for doc in stop_words_removed_doc]
#print(docs_string[0])


In [79]:
MAX_TOKENS = 5000

truncated_docs = [doc[:MAX_TOKENS] for doc in stop_words_removed_doc]
docs_string_truncated = [' '.join(doc) for doc in truncated_docs]

### Creating TF-IDF tables to analyze the most frequently used words

##### The following code is used from UDA Topic Modeling code Demo. 

###### This section covers following things:
###### - Creates two TF-IDF model based on the Document 1 created
###### - Creates one TF-IDF model based on the maximum 2000 token document
###### - Creates two count vectorizer model using Document 1 created - this is just a test to see if this helps fix the issue 

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer

#stop words & token_pattern is added again due to the repeated issue happening 

tfidf_vectorizer1 = TfidfVectorizer(max_df=0.7, min_df=5, stop_words ='english', token_pattern=r'[a-zA-Z]{3,}', max_features= 3000)
tfidf1 = tfidf_vectorizer1.fit_transform(docs_string)

tfidf_vectorizer2 = TfidfVectorizer(max_df=0.8,min_df=10,token_pattern=r'[a-zA-Z]{3,}', stop_words ='english', max_features=5000)
tfidf2 = tfidf_vectorizer2.fit_transform(docs_string)

vocab1 = tfidf_vectorizer1.get_feature_names_out()
vocab2 = tfidf_vectorizer2.get_feature_names_out()

print(f"TF-IDF 1: Vocabulary Length = {len(vocab1)}, TF-IDF Shape = {tfidf1.shape}")
print(f"\nTF-IDF 2: Vocabulary Length = {len(vocab2)}, TF-IDF Shape = {tfidf2.shape}")

TF-IDF 1: Vocabulary Length = 3000, TF-IDF Shape = (153, 3000)

TF-IDF 2: Vocabulary Length = 3030, TF-IDF Shape = (153, 3030)


In [81]:
tfidf_vectorizer_lessTokens = TfidfVectorizer(max_df=0.7, min_df=10, stop_words ='english', token_pattern=r'[a-zA-Z]{3,}', max_features=3000)
tfidf_lessTokens = tfidf_vectorizer_lessTokens.fit_transform(docs_string_truncated)

vocab_lessTokens = tfidf_vectorizer_lessTokens.get_feature_names_out()

print(f"TF-IDF Less Tokens: Vocabulary Length = {len(vocab_lessTokens)}, TF-IDF Shape = {tfidf_lessTokens.shape}")

TF-IDF Less Tokens: Vocabulary Length = 2573, TF-IDF Shape = (153, 2573)


In [82]:
from sklearn.feature_extraction.text import CountVectorizer

# trying CountVextorizer to just test out if this works/fix the issue

count_vectorizer1 = CountVectorizer(max_df=0.7, min_df=5, stop_words ='english', token_pattern=r'[a-zA-Z]{3,}', max_features= 3000)
CV1 = count_vectorizer1.fit_transform(docs_string)

count_vectorizer2 = CountVectorizer(max_df=0.8,min_df=10,token_pattern=r'[a-zA-Z]{3,}', stop_words ='english', max_features=5000)
CV2 = count_vectorizer2.fit_transform(docs_string)

CV1_vocab1 = count_vectorizer1.get_feature_names_out()
CV2_vocab2 = count_vectorizer2.get_feature_names_out()

print(f"CV - 1 : Vocabulary Length = {len(CV1_vocab1)},  Shape = {CV1.shape}")
print(f"\nCV - 2: Vocabulary Length = {len(CV2_vocab2)},  Shape = {CV2.shape}")


CV - 1 : Vocabulary Length = 3000,  Shape = (153, 3000)

CV - 2: Vocabulary Length = 3030,  Shape = (153, 3030)


### Fitting LDA Model to the Data

##### The following code is used from UDA Topic Modeling code Demo. 
###### - Creates two LDA models based on TF-IDF created above
###### - Creates one LDA model based on max 2000 TF-IDF
###### - Creates two LDA models based on the count vectorizer

In [83]:
from sklearn.decomposition import LatentDirichletAllocation

lda1 = LatentDirichletAllocation(n_components = 8, max_iter = 100, doc_topic_prior= 0.01, topic_word_prior = 0.001,
                                 learning_method ='online', learning_offset=50.0, random_state=42)
lda1.fit(tfidf1)

lda2 = LatentDirichletAllocation(n_components = 8, max_iter = 100, doc_topic_prior= 0.01, topic_word_prior = 0.001, 
                                 learning_method ='online', learning_offset=50.0, random_state=42)
lda2.fit(tfidf2)


print (f"LDA 1 Shape: {lda1.components_.shape} & LDA 2 Shape: {lda2.components_.shape}")

# Check if topics are balanced
print("\nLDA 1 topic sums:", lda1.components_.sum(axis=1))
print("LDA 2 topic sums:", lda2.components_.sum(axis=1))

LDA 1 Shape: (8, 3000) & LDA 2 Shape: (8, 3030)

LDA 1 topic sums: [   6.39786721    6.43379478    6.400272   2508.820523      6.41610201
    6.40574775    6.41093099    6.40156833]
LDA 2 topic sums: [   6.46756455    6.46141653    6.456693      6.66499986    6.44042279
    6.45714024    6.46023054 2645.89945466]


In [84]:
lda_lessTokens = LatentDirichletAllocation(n_components = 8, max_iter = 100, doc_topic_prior=0.1, topic_word_prior=0.01,
                    learning_method='online', learning_offset=50.0, random_state=42)
lda_lessTokens.fit(tfidf_lessTokens)


print (f"LDA with Less Token Documents Shape: {lda_lessTokens.components_.shape}")

# Check if topics are balanced
print("\nLDA with Less Token Documents topic sums:", lda_lessTokens.components_.sum(axis=1))

LDA with Less Token Documents Shape: (8, 2573)

LDA with Less Token Documents topic sums: [  28.675584     28.66462407   28.64178601   28.6902225  2541.25328196
   28.61286945   28.62453954   28.63004241]


In [85]:
from sklearn.decomposition import LatentDirichletAllocation

CV1_lda = LatentDirichletAllocation(n_components = 8, max_iter = 100, doc_topic_prior=0.1, topic_word_prior=0.01,
            learning_method='online', learning_offset=50.0, random_state=42)
CV1_lda.fit(CV1)  

CV2_lda = LatentDirichletAllocation(n_components = 8, max_iter = 100, doc_topic_prior=0.1, topic_word_prior=0.01,
            learning_method='online', learning_offset=50.0, random_state=42)
CV2_lda.fit(CV2)  

print(f"LDA 1 Shape: {CV1_lda.components_.shape} & LDA 2 Shape: {CV2_lda.components_.shape}")

# Check if topics are balanced
print("\nLDA 1 topic sums:", CV1_lda.components_.sum(axis=1))
print("LDA 2 topic sums:", CV2_lda.components_.sum(axis=1))


LDA 1 Shape: (8, 3000) & LDA 2 Shape: (8, 3030)

LDA 1 topic sums: [1.46473957e+04 3.55371422e+01 3.51406701e+01 2.80390467e+05
 3.52863973e+01 7.59550275e+04 2.14450957e+02 3.50215929e+01]
LDA 2 topic sums: [3.72385810e+04 3.51701246e+01 3.56779141e+01 3.08952764e+05
 3.52884244e+01 3.62101014e+01 3.58070693e+01 6.66739259e+04]


In [86]:
topic_word_distributions_lt = np.array([row / row.sum() for row in lda_lessTokens.components_])

def print_top_words(tf_1d_table, vocab, num_top_words=15):
    sorted_tuples = sorted(zip(tf_1d_table, vocab), reverse=True)[:num_top_words]
    for _, word in sorted_tuples:
        print(word, end=' ')
    print()

# print the dominant topic (index 5 based on your output)
print("Dominant topic words:")
print_top_words(topic_word_distributions_lt[5], vocab_lessTokens)

# print the other topics too
for i in range(8):
    print(f"[Topic {i}]")
    print_top_words(topic_word_distributions_lt[i], vocab_lessTokens)

Dominant topic words:
moderna availability option waiting detailed violence amazing mrs wide wainwright drop resolution line play police 
[Topic 0]
dilucente pastorek anyways shirt especially cell juvenile honorable ming master april manager audience believe rate 
[Topic 1]
surgery late feel dilucente administration estock practice saturday meal assisted belief offer licensure circulate rfp 
[Topic 2]
dilucente administration suppose richard suffer live employ division pregnancy story spanish develop talotta obtain urban 
[Topic 3]
shackle implementation book life force solution example safety manner april detox spread intend partnership unit 
[Topic 4]
dilucente incarcerated unit connor chief kind tablet force administration resident update believe allow answer hospital 
[Topic 5]
moderna availability option waiting detailed violence amazing mrs wide wainwright drop resolution line play police 
[Topic 6]
contraband elect incarcerated federal consider read apologize medium november typ

## LDA Topic Modeling — Process Notes & Challenges

### What Was Tried

**Preprocessing iterations:**
- Removed blank documents (3 documents: indices 34, 43, 44)
- Removed numbers and digits, keeping only 4-digit years
- Applied custom stop words list (iteratively expanded over multiple runs) covering:
  - Conversational fillers (okay, yeah, think, know, etc.)
  - Single letters appearing as artifacts
  - Formal meeting procedural words (motion, second, approve, etc.)
  - Geographic boilerplate (allegheny, county, pittsburgh)
  - Board member and staff names (based on the output of different models)
- Tried document truncation to 2000 & 5000 tokens to address document length imbalance

**Vectorization approaches tried:**
- TF-IDF with multiple parameter combinations (max_df: 0.5-0.9, min_df: 5-15, max_features: 500-5000)
- CountVectorizer as an alternative to TF-IDF
- token_pattern restricted to letters only, minimum 3 characters
  
**LDA configurations tried:**
- Number of topics: 3, 5, 8, 10
- max_iter: 30, 50, 100, 500 (some might not be there now, since they have been tested and weren't find to be helpful)
- Varying doc_topic_prior (0.01, 0.1) and topic_word_prior (0.001, 0.01)
- learning_offset: 50.0
- 30 total model combinations across 10 TF-IDF configurations in the older code notebook
- 5 being tested in this notebook

---

### Persistent Issue: Topic Collapse

Throughout all iterations, LDA consistently produced **topic collapse** — where one topic dominates the probability distribution while all others are near-zero. For example:

- LDA with Less Token Documents topic sums:28.675584, 28.66462407, 28.64178601, 28.6902225, 2541.25328196,28.61286945, 28.62453954, 28.63004241

So, LDA is assigning nearly all words (probability) to a single topic (such as 2541 for one topics as compared to 28 for rest) instead of trying to find distinct theme.

### Most Reasonable (sane T_T) Output
After trying different models, most reasonable output is observed when the documents were truncated to 2000/5000 (both were tested) tokens.